## 實驗：指標選擇之消融研究 (Ablation Study for Feature Selection)

### Subtask:
根據作業要求，我們需要驗證加入 `Volume` 和 `MA5` 這些技術指標的有效性。以下程式碼將會自動跑不同特徵組合的預測結果，並比較其 RMSE 與 MAPE，以佐證我們最終選擇 `[Close, Volume, MA5]` 的合理性。

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Layer, Dropout
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
import tensorflow as tf
import itertools
import yfinance as yf
from datetime import datetime, timedelta

I0000 00:00:1775841220.694212 3776277 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
tf.random.set_seed(SEED)

In [ ]:
def fetch_stock_data(ticker):
    """
    Fetches historical stock data for a given ticker for the past 10 years.

    Args:
        ticker (str): The stock ticker symbol (e.g., '2330.TW' for TSMC).

    Returns:
        pandas.DataFrame: Historical stock data including Open, High, Low, Close, Volume, etc.
                          Returns an empty DataFrame if data fetching fails.
    """
    end_date = pd.to_datetime("2026-03-20") + timedelta(days=1)
    start_date = end_date - timedelta(days=10*365) # Approximate 10 years

    try:
        stock = yf.Ticker(ticker)
        # Fetch daily data for the specified period
        data = stock.history(start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='1d')
        if data.empty:
            print(f"No data found for {ticker} in the last 10 years.")
        else:
            print(f"Successfully fetched data for {ticker} from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}.")
        return data
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return pd.DataFrame() # Return an empty DataFrame on error

stock_df = fetch_stock_data('2330.TW')

stock_df["MA5"] = stock_df["Close"].rolling(5).mean()
stock_df = stock_df.dropna(subset=["MA5"])
print(f"Data fetched! Shape: {stock_df.shape}")

class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], input_shape[-1]),
                               initializer='random_normal', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[-1],),
                               initializer='zeros', trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, x):
        ui = K.tanh(K.dot(x, self.W) + self.b)
        uw = K.ones_like(ui[:, :, 0]) 
        uw = K.expand_dims(uw)
        alpha = K.softmax(K.sum(ui * uw, axis=2))
        alpha = K.expand_dims(alpha)
        context_vector = K.sum(x * alpha, axis=1)
        return context_vector

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

def mean_absolute_percentage_error_fn(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def run_feature_experiment_rolling(feature_cols, target_date_str="2026-03-20", test_size=0.05, epochs=50, batch_size=32, _look_back=60, dropout_rate=0.0):
    print(f"--- 測試特徵組合: {feature_cols} | look_back: {_look_back} | Dropout: {dropout_rate} ---")
    
    # 找出對應日期的 index
    target_dt = pd.to_datetime(target_date_str).tz_localize(stock_df.index.tz)
    
    if target_dt not in stock_df.index:
        valid_dates = stock_df.index[stock_df.index <= target_dt]
        if len(valid_dates) == 0:
            raise ValueError(f"找不到 {target_date_str} 之前的資料。")
        target_dt = valid_dates[-1]
        print(f"找不到 {target_date_str} 確切日期，使用最近的交易日: {target_dt.date()}")
        
    target_idx_num = stock_df.index.get_loc(target_dt)
    
    # 我們需要截至 target_dt 為止的資料
    data_idx_end = target_idx_num + 1
    data_df = stock_df.iloc[:data_idx_end].copy()
    
    scalers = {}
    scaled_feature_list = []
    
    for col in feature_cols:
        scaler = MinMaxScaler(feature_range=(0, 1))
        # 在原始 final code 中，資料切割前就呼叫了 fit_transform，為了對齊我們這裡直接 fit 整個 data_df
        scaled_feat = scaler.fit_transform(data_df[col].values.reshape(-1, 1))
        scalers[col] = scaler
        scaled_feature_list.append(scaled_feat)
        
    scaled_features = np.hstack(scaled_feature_list)
    target_idx = feature_cols.index("Close")
    
    # 建立 Dataset (X: 歷史 look_back 天, y: 第 look_back 天的 Close)
    def create_dataset_exp(dataset, lb):
        dX, dY = [], []
        for i in range(len(dataset) - lb):
            dX.append(dataset[i:(i + lb), :])
            dY.append(dataset[i + lb, target_idx])
        return np.array(dX), np.array(dY)
        
    X_exp, y_exp = create_dataset_exp(scaled_features, _look_back)
    
    # 使用 train_test_split 對齊 final code 的 0.05 測試集設定
    X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(X_exp, y_exp, test_size=test_size, shuffle=False)
    
    # 建構 Attention-LSTM
    model_e = Sequential()
    model_e.add(Input(shape=(_look_back, len(feature_cols))))
    
    model_e.add(LSTM(128, return_sequences=True))
    if dropout_rate > 0:
        model_e.add(Dropout(dropout_rate))
    model_e.add(LSTM(64, return_sequences=True))
    if dropout_rate > 0:
        model_e.add(Dropout(dropout_rate))
    model_e.add(Attention())
    if dropout_rate > 0:
        model_e.add(Dropout(dropout_rate))
    model_e.add(Dense(1))
    
    model_e.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    
    # 訓練模型
    model_e.fit(X_train_e, y_train_e, epochs=epochs, batch_size=batch_size, verbose=0)
    
    # 對測試集進行預測
    test_pred_e = model_e.predict(X_test_e, verbose=0)
    
    # 反正規化
    target_scaler = scalers["Close"]
    pred_inv = target_scaler.inverse_transform(test_pred_e).flatten()
    actual_inv = target_scaler.inverse_transform(y_test_e.reshape(-1, 1)).flatten()
    
    # 計算 RMSE 與 MAPE
    rmse = np.sqrt(mean_squared_error(actual_inv, pred_inv))
    mape = mean_absolute_percentage_error_fn(actual_inv, pred_inv)
    
    num_test_days = len(y_test_e)
    dates_to_predict = data_df.index[-num_test_days:].strftime('%Y-%m-%d').tolist()
    print(f"對最後 {num_test_days} 天 ({dates_to_predict[0]} 到 {dates_to_predict[-1]}) 進行預測:")
    print(f"RMSE: {rmse:.4f} | MAPE: {mape:.2f}%\n")
    
    return rmse, mape

# 定義變因選項
feature_options = [
    ["Close"],
    ["Close", "Volume"],
    ["Close", "MA5"],
    ["Close", "Volume", "MA5"]
]
look_back_options = [60, 90, 120]
dropout_options = [0.0, 0.2]

# 自動生成全網格實驗組合
experiments = []
for feats, lb, drop in itertools.product(feature_options, look_back_options, dropout_options):
    # 格式化實驗名稱
    feat_name = ", ".join(feats).replace("Volume", "Vol")
    exp_name = f"{feat_name} (LB={lb}, Drop={drop})"
    
    experiments.append({
        "feats": feats, 
        "look_back": lb, 
        "dropout_rate": drop,
        "name": exp_name
    })

results = []
target_date_str = "2026-03-20"
test_size = 0.05

print(f"=== 啟動全網格實驗 ({len(experiments)} 組)：以 test_size={test_size} ({test_size*100}%) 作為測試集 ===")
for i, exp in enumerate(experiments, 1):
    print(f"\n[實驗 {i}/{len(experiments)}]")
    rmse, mape = run_feature_experiment_rolling(
        feature_cols=exp["feats"], 
        target_date_str=target_date_str, 
        test_size=test_size, 
        epochs=50,
        _look_back=exp["look_back"],
        dropout_rate=exp["dropout_rate"]
    )
    results.append({
        "Experiment": exp["name"],
        "Test RMSE": round(rmse, 4),
        "Test MAPE (%)": round(mape, 2)
    })

# 顯示結果 DataFrame 並依據 MAPE 進行排序，方便看出最佳組合
results_df = pd.DataFrame(results).sort_values("Test MAPE (%)")
print(f"\n=== 實驗結果匯總 (依 MAPE 排序) ===")
print(results_df.to_markdown(index=False))

Successfully fetched data for 2330.TW from 2016-03-23 to 2026-03-21.
Data fetched! Shape: (2425, 8)
=== 啟動全網格實驗 (24 組)：以 test_size=0.05 (5.0%) 作為測試集 ===

[實驗 1/24]
--- 測試特徵組合: ['Close'] | look_back: 60 | Dropout: 0.0 ---


I0000 00:00:1775841223.558964 3776277 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38484 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:01:00.0, compute capability: 8.0
I0000 00:00:1775841227.181852 3776903 cuda_dnn.cc:461] Loaded cuDNN version 91002


對最後 119 天 (2025-09-17 到 2026-03-20) 進行預測:
RMSE: 46.0920 | MAPE: 2.23%


[實驗 2/24]
--- 測試特徵組合: ['Close'] | look_back: 60 | Dropout: 0.2 ---
對最後 119 天 (2025-09-17 到 2026-03-20) 進行預測:
RMSE: 108.6681 | MAPE: 5.90%


[實驗 3/24]
--- 測試特徵組合: ['Close'] | look_back: 90 | Dropout: 0.0 ---
對最後 117 天 (2025-09-19 到 2026-03-20) 進行預測:
RMSE: 37.1420 | MAPE: 1.81%


[實驗 4/24]
--- 測試特徵組合: ['Close'] | look_back: 90 | Dropout: 0.2 ---
對最後 117 天 (2025-09-19 到 2026-03-20) 進行預測:
RMSE: 55.0783 | MAPE: 2.65%


[實驗 5/24]
--- 測試特徵組合: ['Close'] | look_back: 120 | Dropout: 0.0 ---
對最後 116 天 (2025-09-22 到 2026-03-20) 進行預測:
RMSE: 47.7786 | MAPE: 2.23%


[實驗 6/24]
--- 測試特徵組合: ['Close'] | look_back: 120 | Dropout: 0.2 ---
對最後 116 天 (2025-09-22 到 2026-03-20) 進行預測:
RMSE: 105.9010 | MAPE: 5.58%


[實驗 7/24]
--- 測試特徵組合: ['Close', 'Volume'] | look_back: 60 | Dropout: 0.0 ---
對最後 119 天 (2025-09-17 到 2026-03-20) 進行預測:
RMSE: 60.5259 | MAPE: 3.15%


[實驗 8/24]
--- 測試特徵組合: ['Close', 'Volume'] | look_back: 60 | Dropout: 0.2 ---
對最後

: 

In [ ]:
os._exit(0)

: 